# PESO replication: does a continual-LoRA regularizer beat plain continual LoRA?

PESO (Yoo et al., ICLR 2026, [arXiv 2510.25093](https://arxiv.org/abs/2510.25093)) continually updates a Llama-3.2-1B generative recommender one chronological block at a time. It uses one LoRA adapter plus a penalty that keeps the adapter close to the previous block's. We reran its released Instruments setup with three seeds. This notebook reads only the **committed metric files** in `replication/peso/results/`; it trains and evaluates nothing.

These are results on PESO's released Instruments blocks, not the project's Electronics/Movies & TV data. Protocol, the two environment deviations (Python 3.10; one GPU with gradient accumulation 4) and caveats are in [`docs/PESO-REPLICATION.md`](../docs/PESO-REPLICATION.md).

In [ ]:
import importlib.util
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
spec = importlib.util.spec_from_file_location("summarize", root / "replication/peso/summarize.py")
summarize = importlib.util.module_from_spec(spec)
spec.loader.exec_module(summarize)

runs = [root / "replication/peso/results" / f"seed{s}" for s in (42, 43, 44)]
table = summarize.summarize(runs)
print(summarize.markdown(table))

## Table 2 against the paper

Each cell is the mean ± sd across seeds, with the paper's value in brackets. The paper averages D2–D4 (code blocks 1–3). Pretrain and single evolving LoRA land within about 5% of the paper. PESO's NDCG@10 is about 10% lower, so the paper's ordering single LoRA < PESO does not reproduce.

## Per block

Block 0 is the initial-training block. PESO and single LoRA share each seed's block-0 checkpoint, so their block-0 scores must be identical, which serves as a sanity check.

In [ ]:
metric = "ndcg@10"
print(f"{metric} by test block (each method is evaluated on block j after training through block j)")
print("| Seed | Method | " + " | ".join(f"b{j}" for j in range(5)) + " |")
print("| --- | --- |" + " --- |" * 5)
for seed, run in zip((42, 43, 44), runs):
    for method, tag in summarize.TAGS.items():
        blocks = summarize.per_block(run / "results" / tag)
        print(f"| {seed} | {method} | " + " | ".join(f"{blocks[j][metric]:.4f}" for j in range(5)) + " |")

## Paired difference: PESO − single evolving LoRA

Both methods start from the same block-0 checkpoint and see the same data in each seed, so their per-seed difference is far less noisy than either average.

In [ ]:
diff = summarize.paired(runs, "lora", "peso")
print("| Metric | per seed (42, 43, 44) | mean ± sd | paper |")
print("| --- | --- | --- | --- |")
for m, d in diff.items():
    per_run = ", ".join(f"{x:+.4f}" for x in d["per_run"])
    print(f"| {m} | {per_run} | {d['mean']:+.4f} ± {d['sd']:.4f} | {d['paper']:+.4f} |")

## How large is seed noise?

Single LoRA's own improvement over Pretrain changes sign across seeds. A single-seed table, as in the paper, cannot separate a gain of about 0.001 from this variation.

In [ ]:
noise = summarize.paired(runs, "pretrain", "lora")
for m in ("ndcg@10", "hit@10"):
    d = noise[m]
    print(f"single LoRA - Pretrain, {m}: " + ", ".join(f"{x:+.4f}" for x in d["per_run"]) + f"  (paper {d['paper']:+.4f})")

## Transfer matrix: every saved state on every block (seed 42)

Table 2 scores each block only right after training on it. Here every saved state (the model after training block *i*) is evaluated on every block's test users, so the matrix shows how earlier blocks fare as training continues.

- **Evaluator:** ours (`replication/peso/peso_eval.py`), which matches upstream's exactly on the same GPU, one user at a time.
- **Settings:** 20 beams, so @20 metrics exist, and at most 2,500 seeded test users per block. Values are therefore not directly comparable with Table 2's 10-beam, all-user numbers.
- **Source:** `replication/peso/results/seed42/reports/transfer.json`, counts only, written by `replication/peso/transfer_report.py` from per-user ranked lists that stay on the run volume.
- **Backward transfer:** the mean over blocks 0–3 of (score after the last block − score right after training that block). Negative means earlier blocks got worse.

In [ ]:
import json

transfer = json.loads((root / "replication/peso/results/seed42/reports/transfer.json").read_text())
for method in ("lora", "peso"):
    m = transfer["methods"][method]["k10"]
    print(f"{method}: NDCG@10; row = state after training block i, column = test block j")
    print("| state | " + " | ".join(f"b{j}" for j in m["matrix"]["blocks"]) + " |")
    print("| --- |" + " --- |" * len(m["matrix"]["blocks"]))
    for state, row in zip(m["matrix"]["states"], m["matrix"]["values"]):
        print(f"| {state} | " + " | ".join(f"{x:.4f}" for x in row) + " |")
    k20 = transfer["methods"][method]["k20"]["backward_transfer"]
    print(f"backward transfer: NDCG@10 {m['backward_transfer']:+.4f}, NDCG@20 {k20:+.4f}\n")

## The counter: hits kept, lost and gained at each update

A test user is a hit when their target item is in the top 10. Between consecutive states, on each block's fixed users (all five blocks, including ones not trained on yet), each earlier hit is either **kept** or **lost**, and each earlier miss is either **gained** or stays a miss. Averages hide this churn: a block's NDCG can stay flat while many individual users swap places.

Groups follow the paper's Table 3: *new* users first appear in the test block, *continuing* users were in the previous block, and *dormant* users skipped it and returned. Groups exist only for blocks 1–4 (block 0 has no earlier block), and users with fewer than three interactions in the block are unlabelled, so the group rows do not add up to *all*.

In [ ]:
groups = ("all", "new", "continuing", "dormant")
print("| method | group | kept | lost | gained | share of earlier hits lost |")
print("| --- | --- | --- | --- | --- | --- |")
for method in ("lora", "peso"):
    steps = transfer["methods"][method]["k10"]["consecutive_flips"].values()
    for group in groups:
        kept, lost, gained = (sum(s[group][key] for s in steps if group in s) for key in ("kept", "lost", "gained"))
        print(f"| {method} | {group} | {kept} | {lost} | {gained} | {lost / (kept + lost):.0%} |")

## Head to head: PESO against single LoRA on the same users

Same saved state index, same block, same users. *PESO-only* users are top-10 hits for PESO and misses for single LoRA; *LoRA-only* is the reverse. McNemar's exact test asks whether the two counts differ by more than chance.

In [ ]:
pairs = transfer["head_to_head"]["k10"]
print("| state, block | users | PESO-only hits | LoRA-only hits | McNemar p |")
print("| --- | --- | --- | --- | --- |")
for name, counts in pairs.items():
    a = counts["all"]
    print(f"| {name} | {a['users']} | {a['gained']} | {a['lost']} | {a['mcnemar_p']:.2f} |")
print(f"smallest p over {len(pairs)} pairs: {min(c['all']['mcnemar_p'] for c in pairs.values()):.2f}")

## Why PESO tracks single LoRA: the size of its penalty

Upstream's PESO penalty is λ = 2.0 times the mean, over all 112 LoRA modules, of KL(softmax(A) ‖ softmax(A_prev)) plus the same for B. Here A is the flattened adapter matrix and A_prev the previous block's snapshot. Each softmax runs over 4,096–65,536 entries of small numbers, so both distributions are almost uniform. For small changes the KL is then about ‖Δ − mean(Δ)‖² / (2n): dividing by n shrinks it.

`replication/peso/penalty_size.py` recomputes the penalty from the saved seed-42 checkpoints at the end of each block, where it is largest because each block starts with the adapter equal to the snapshot. For scale, the seed-42 training log shows a recommendation loss of about 2.0 and gradient norms of 6–11.

In [ ]:
penalty = json.loads((root / "replication/peso/results/seed42/reports/penalty.json").read_text())
print("| block | penalty (float32) | small-change approximation | gradient norm | penalty in bfloat16 | adapter change |")
print("| --- | --- | --- | --- | --- | --- |")
for block, p in penalty["blocks"].items():
    print(f"| {block} | {p['penalty_float32']:.2e} | {p['penalty_small_change_approx']:.2e} | "
          f"{p['gradient_norm_float32']:.2e} | {p['penalty_bfloat16']:.1e} | {p['adapter_change_relative']:.1%} |")

The penalty stays below a millionth of the loss, and its gradient is about 10⁻⁷ times the training gradient, although the adapter moves 4–5% per block. In bfloat16, the dtype the model trains in, the computed value is mostly rounding error. In the released code, then, PESO trains almost exactly like single LoRA with the same seed. That matches the near-identical results above. The paper's own implementation may differ; this is a statement about the released code at commit `edbec5f`.

## Inside the model: a Jacobian-lens probe (seed 42)

The Jacobian lens (J-lens; Gurnee et al., 2026) reads which output token a middle layer is preparing to produce. For layer ℓ it uses the rows of W_U·J_ℓ, where J_ℓ is the Jacobian of the final residual stream with respect to layer ℓ. The logit lens is the special case J = I and serves as the baseline.

Our adaptation (`replication/peso/jlens.py`):
- **Where it reads:** only at the decision point, where the first item code is generated, and only for the first of the 4 semantic-ID tokens. That is 256 candidate first codes.
- **Inputs:** J is averaged over 200 fixed test prompts. Readouts use 500 test users per block.
- **Score:** MRR is the mean of 1/rank of the target's first code among the 256; 1.0 means always ranked first.

In [ ]:
jlens = json.loads((root / "replication/peso/results/seed42/reports/jlens.json").read_text())
lenses = ["final", "jlens_l4", "jlens_l8", "jlens_l12", "logit_l4", "logit_l8", "logit_l12"]
print("Readout quality after the last block (state 4): MRR / share ranked in the top 10 of 256")
print("| method | " + " | ".join(lenses) + " |")
print("| --- |" + " --- |" * len(lenses))
for method, d in jlens["methods"].items():
    q = d["readout_quality"]["state4"]
    print(f"| {method} | " + " | ".join(f"{q[x]['mrr']:.3f} / {q[x]['top10']:.0%}" for x in lenses) + " |")

print("\nFrom each block's own state to state 4 (blocks 1-3, 500 users each): output vs mid-layer rank change")
print("| method | layer | Spearman (output, J-lens) | output worse but J-lens not |")
print("| --- | --- | --- | --- |")
for method, d in jlens["methods"].items():
    for layer, v in d["internal_vs_output"].items():
        print(f"| {method} | {layer} | {v['spearman_final_vs_mid']:.2f} | {v['share_output_worse_but_mid_not']:.0%} |")

print("\nJ-lens drift at state 4, relative to the block-0 lens: total, item-head part (U_i - U_0) J_i, inner-layer part U_0 (J_i - J_0)")
print("| method | layer | total | item head | inner layers |")
print("| --- | --- | --- | --- | --- |")
for method, d in jlens["methods"].items():
    for layer, v in d["drift_split"]["state4"].items():
        print(f"| {method} | {layer} | {v['total_rel']:.3f} | {v['head_rel']:.3f} | {v['pathway_rel']:.3f} |")

What the probe shows on this seed:

- **The J-lens reads the coming recommendation from early layers.** It ranks the target's first code about 4× better than the logit lens at layer 4 and nearly 3× better at layer 8, roughly half as well as the final output.
- **Internal and output changes move together.** For about 11% of users the output rank of an old block's target worsened while the mid-layer J-lens rank did not: limited evidence of internal retention beneath output forgetting.
- **The lens drifts steadily, through both the item head and the inner layers.** PESO's drift equals single LoRA's to three decimals at every state, as the penalty measurement predicts.

**Limits.** This is one seed and the first code token only, which is coarse (256 groups) and so undercounts item-level forgetting. The two drift parts are vectors, so their sizes do not add up to the total. Workspace properties were shown on much larger models; this is exploratory and not a claim about PESO's design.

## O8: does a penalty that actually acts help? (seed 42)

PESO was retrained with one change: the KL penalty scaled by each matrix's size, at λ = 400, which calibration showed pulls about as hard as the task loss. It was then evaluated on the pairs that backward transfer and new-block accuracy need, with the same users and settings as the transfer matrix. The report is `replication/peso/results/seed42/reports/o8_peso-size-w400.json`.

In [ ]:
o8 = json.loads((root / "replication/peso/results/seed42/reports/o8_peso-size-w400.json").read_text())
labels = {"lora": "single LoRA", "peso": "released PESO", o8["variant"]: "PESO, size-normalized λ=400"}
print("| NDCG@10 | backward transfer | old blocks after block 4 | own state, blocks 1-3 | own state, blocks 1-4 |")
print("| --- | --- | --- | --- | --- |")
for method, label in labels.items():
    s = o8["methods"][method]["ndcg@10"]
    print(f"| {label} | {s['backward_transfer']:+.4f} | {s['last_state_old_blocks_mean']:.4f} | "
          f"{s['own_state_blocks_1_3']:.4f} | {s['own_state_blocks_1_4']:.4f} |")
old = [o8["head_to_head"]["k10"][f"state4_block{b}"]["all"] for b in range(4)]
gained, lost = sum(a["gained"] for a in old), sum(a["lost"] for a in old)
print(f"\nOld blocks after block 4, top-10: {gained} users right only with the penalty, {lost} only without")

- **A penalty that acts reduces forgetting a little.** Backward transfer is −0.0029 against −0.0038, and old blocks end about 8% higher.
- **It doesn't raise Table 2's own-state score.**
- **One seed, not significant.** User by user the gain is not significant (exact test p = 0.28 pooled over the old blocks).
- **The J-lens shows why.** The inner-layer drift shrinks by 26–50%. The item-head drift tied to code 1 does not change, and the newest block's exact-item codes are learned less well. See `docs/PESO-REPLICATION.md`.